# Pipeline parallelism: 1F1B

Run one model stage per `flock` rank with a non-interleaved 1F1B schedule. Activations move to the next rank during the forward pass; activation gradients move to the previous rank during the backward pass. Optimizer steps happen only after the pipeline drains.

For rank $r$, pipeline depth $P$, and $M$ microbatches, the number of warmup forwards is

$$
\min(P-r-1, M).
$$

The remaining work alternates one forward and one backward before the cooldown backward passes.

In [1]:
import copy

import flock
import flock.torch as ft
import torch

from dist_puzzles import check_comm

In [2]:
async def pipeline_1f1b(
    stage: torch.nn.Module,
    X: torch.Tensor,
    target: torch.Tensor,
    microbatch_count: int,
    learning_rate: float,
) -> tuple[list[tuple[str, int]], tuple[torch.Tensor, ...], torch.Tensor | None]:
    rank = flock.get_rank()
    world_size = flock.get_world_size()
    if microbatch_count < 1:
        raise ValueError(f"microbatch_count must be positive, got {microbatch_count}")
    if target.shape[0] != X.shape[0]:
        raise ValueError(
            f"input and target batch sizes must match, got {X.shape[0]} and {target.shape[0]}"
        )
    if X.shape[0] % microbatch_count:
        raise ValueError(
            f"batch size {X.shape[0]} must be divisible by microbatch_count={microbatch_count}"
        )

    inputs = X.chunk(microbatch_count)
    targets = target.chunk(microbatch_count)
    optimizer = torch.optim.SGD(stage.parameters(), lr=learning_rate)
    optimizer.zero_grad(set_to_none=True)

    activations: dict[int, torch.Tensor] = {}
    outputs: dict[int, torch.Tensor] = {}
    final_outputs: dict[int, torch.Tensor] = {}
    events: list[tuple[str, int]] = []

    async def forward(microbatch: int) -> None:
        if rank == 0:
            activation = inputs[microbatch]
        else:
            received = await ft.recv(rank - 1, tag=microbatch)
            activation = received.detach().requires_grad_()

        output = stage(activation)
        activations[microbatch] = activation
        outputs[microbatch] = output

        if rank == world_size - 1:
            final_outputs[microbatch] = output.detach().clone()
        else:
            await ft.isend(rank + 1, output, tag=microbatch).wait()

        events.append(("F", microbatch))

    async def backward(microbatch: int) -> None:
        output = outputs.pop(microbatch)
        activation = activations.pop(microbatch)

        if rank == world_size - 1:
            loss = torch.nn.functional.mse_loss(output, targets[microbatch])
            (loss / microbatch_count).backward()
        else:
            output_grad = await ft.recv(rank + 1, tag=microbatch)
            output.backward(output_grad)

        if rank > 0:
            activation_grad = activation.grad
            assert activation_grad is not None
            await ft.isend(rank - 1, activation_grad, tag=microbatch).wait()

        events.append(("B", microbatch))

    warmup = min(world_size - rank - 1, microbatch_count)
    remaining = microbatch_count - warmup

    for microbatch in range(warmup):
        await forward(microbatch)

    for microbatch in range(remaining):
        await forward(microbatch + warmup)
        await backward(microbatch)

    for microbatch in range(remaining, microbatch_count):
        await backward(microbatch)

    optimizer.step()
    parameters = tuple(parameter.detach().clone() for parameter in stage.parameters())
    output = (
        torch.cat([final_outputs[microbatch] for microbatch in range(microbatch_count)])
        if rank == world_size - 1
        else None
    )
    return events, parameters, output

## Validation

Check the local operation order, compare the updated parameters and output with a sequential PyTorch step, and require communication to use adjacent pipeline links only.

In [3]:
def make_stages(world_size: int, width: int) -> list[torch.nn.Module]:
    if world_size < 1:
        raise ValueError(f"world_size must be positive, got {world_size}")

    stages = []
    for rank in range(world_size):
        layers: list[torch.nn.Module] = [torch.nn.Linear(width, width)]
        if rank < world_size - 1:
            layers.append(torch.nn.GELU())
        stages.append(torch.nn.Sequential(*layers))
    return stages


def expected_1f1b_events(
    rank: int,
    world_size: int,
    microbatch_count: int,
) -> list[tuple[str, int]]:
    warmup = min(world_size - rank - 1, microbatch_count)
    remaining = microbatch_count - warmup
    events = [("F", microbatch) for microbatch in range(warmup)]
    for microbatch in range(remaining):
        events.append(("F", microbatch + warmup))
        events.append(("B", microbatch))
    events.extend(("B", microbatch) for microbatch in range(remaining, microbatch_count))
    return events

In [4]:
def check_pipeline(world_size: int, microbatch_count: int) -> flock.Tracer:
    width = 8
    microbatch_size = 2
    learning_rate = 0.05

    torch.manual_seed(1_000 + world_size)
    initial_stages = make_stages(world_size, width)
    pipeline_stages = copy.deepcopy(initial_stages)
    reference_stages = copy.deepcopy(initial_stages)

    generator = torch.Generator().manual_seed(2_000 + world_size)
    batch_size = microbatch_count * microbatch_size
    X = torch.randn(batch_size, width, generator=generator)
    target = torch.randn(batch_size, width, generator=generator)

    distributed = flock.distribute(
        workers=world_size,
        seed=0,
        trace=True,
    )(pipeline_1f1b)
    results, trace = distributed(
        flock.per_rank(pipeline_stages),
        X,
        target,
        microbatch_count,
        learning_rate,
    )

    reference = torch.nn.Sequential(*reference_stages)
    reference_optimizer = torch.optim.SGD(reference.parameters(), lr=learning_rate)
    reference_optimizer.zero_grad(set_to_none=True)
    reference_output = reference(X)
    torch.nn.functional.mse_loss(reference_output, target).backward()
    reference_optimizer.step()

    for rank, (events, parameters, output) in enumerate(results):
        assert events == expected_1f1b_events(rank, world_size, microbatch_count)
        for actual, expected in zip(
            parameters,
            reference_stages[rank].parameters(),
            strict=True,
        ):
            torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)

        if rank == world_size - 1:
            assert output is not None
            torch.testing.assert_close(output, reference_output.detach())
        else:
            assert output is None

    links = {
        link
        for rank in range(world_size - 1)
        for link in ((rank, rank + 1), (rank + 1, rank))
    }
    check_comm(
        trace,
        total_messages=2 * microbatch_count * (world_size - 1),
        allowed_links=links,
        required_links=links,
    )
    assert trace.summary().collective_counts == ()
    return trace

In [5]:
cases = ((1, 3), (2, 5), (4, 8), (6, 3))

for pipeline_depth, microbatch_count in cases:
    trace = check_pipeline(pipeline_depth, microbatch_count)
    print(
        f"depth={pipeline_depth}, microbatches={microbatch_count}, "
        f"messages={trace.summary().total_messages}"
    )

depth=1, microbatches=3, messages=0
depth=2, microbatches=5, messages=10
depth=4, microbatches=8, messages=48
depth=6, microbatches=3, messages=30
